# Sugarscape Economy Notebook (Colab Ready)

This notebook runs the upgraded Sugarscape simulation with economy, banks/loans, markets, trust/knowledge, elections, firms, and animations.


In [ ]:
# Colab setup (safe to re-run)
import os, subprocess, math
from pathlib import Path

print('Python:', os.sys.version.split()[0])
print('CPU cores visible:', os.cpu_count())

# Optional: install plotting extras in Colab runtimes
try:
    import matplotlib  # noqa
except Exception:
    !pip -q install matplotlib pillow


In [ ]:
# Hardware-aware worker estimator (CPU + GPU)
import os, re, subprocess

def detect_gpu_info():
    try:
        out = subprocess.check_output([
            'nvidia-smi',
            '--query-gpu=name,memory.total',
            '--format=csv,noheader,nounits'
        ], text=True).strip().splitlines()
        gpus = []
        for line in out:
            name, mem = [x.strip() for x in line.split(',')]
            gpus.append({'name': name, 'memory_gb': float(mem)/1024.0})
        return gpus
    except Exception:
        return []


def suggested_workers(cpu_bound=True):
    cores = os.cpu_count() or 2
    # leave one core for notebook UI
    cpu_workers = max(1, cores - 1)
    gpus = detect_gpu_info()
    if not gpus:
        return cpu_workers, {'gpu': 'none'}

    # Sim is mostly CPU-bound; GPU class can still guide conservative parallelism.
    name = gpus[0]['name'].upper()
    mem = gpus[0]['memory_gb']

    if 'A100' in name:
        gpu_workers = 8 if mem >= 40 else 6
    elif 'L5' in name:
        gpu_workers = 4
    else:
        gpu_workers = max(2, min(8, int(mem // 6)))

    return min(cpu_workers, gpu_workers) if cpu_bound else gpu_workers, gpus

workers, gpu_info = suggested_workers(cpu_bound=True)
print('Detected GPUs:', gpu_info)
print('Suggested worker processes:', workers)


In [ ]:
# Import simulation module
import dataclasses
from pathlib import Path

import sugarscape_sim as ss


In [ ]:
# Run one simulation with snapshots + create animation
out_dir = Path('colab_outputs')
out_dir.mkdir(exist_ok=True)

cfg = ss.SimulationConfig(steps=120, seed=42)
sim = ss.Simulation(cfg)
history, snapshots = sim.run_with_snapshots(every=1)

anim_ok = ss.animate_snapshots(
    snapshots,
    out_dir / 'dynamic_switching.mp4',
    title='Dynamic Switching Economy',
    fps=12,
)

print('frames:', len(snapshots), 'animation_written:', anim_ok)
print('final alive:', history['alive'][-1] if history['alive'] else 0)


In [ ]:
# Build animations for each experiment profile
profiles = {
    'baseline_growth': dict(dynamic_switching=False, fixed_lambda=1.0),
    'baseline_survival': dict(dynamic_switching=False, fixed_lambda=0.0),
    'dynamic_switching': dict(dynamic_switching=True),
    'floor_intervention': dict(dynamic_switching=True, floor_intervention=True),
    'stress_test': dict(dynamic_switching=True, shock_step=60, shock_severity=0.65),
}

all_hist = {}
for name, overrides in profiles.items():
    cfg = dataclasses.replace(ss.SimulationConfig(steps=120, seed=42), **overrides)
    sim = ss.Simulation(cfg)
    hist, snaps = sim.run_with_snapshots(every=1)
    all_hist[name] = hist
    ss.animate_snapshots(snaps, out_dir / f'{name}.mp4', title=name, fps=12)
    print(f'{name}: frames={len(snaps)} alive_end={hist["alive"][-1] if hist["alive"] else 0}')

ss.write_csv(all_hist, out_dir)
print('CSV + animations written to', out_dir.resolve())


In [ ]:
# Preview files in notebook
import os
for fn in sorted(os.listdir('colab_outputs')):
    print(fn)
